# VISIT_OCCURRENCE

See [visit_occurrence](https://ohdsi.github.io/CommonDataModel/cdm54.html#visit_occurrence).

This table records each patient's encounters with the healthcare system. It is somewhat like the fine detail of the observation_period table. The fields in the table are as follows:

```{mermaid}
erDiagram
    OMOP_VISIT_OCCURRENCE {
        integer visit_occurence_id
        integer person_id
        integer visit_concept_id
        varchar(50) visit_source_value
        integer visit_source_concept_id
        date visit_start_date
        datetime visit_start_datetime
        date visit_end_date
        datetime visit_end_datetime
        integer visit_type_concept_id
        integer provider_id
        integer care_site_id
        integer admitted_from_concept_id
        varchar(50) admitted_from_source_value
        integer discharged_to_concept_id
        varchar(50) discharged_to_source_value
        integer preceding_visit_occurrence_id
    }
```

The script that handles the transformation from the original file to OMOP format is [genomop_visit.py](../examples/genomop_visit.py). In this case we have to gather all those files that contain information about a patient's interaction with the health system. Not all files have this info, hence, not all files need to be taken into account.

Each interaction will have its own [visit_concept_id](https://athena.ohdsi.org/search-terms/terms?domain=Visit&conceptClass=Visit&page=1&pageSize=15), which will need to be assigned. The easier way is to apply the same code to every record in the file, but sometimes that cannot be done. See the detail parameter description for more information.

The configuration file will be [genomop_visit_params.yaml](../examples/genomop_visit_params.yaml). The configuration file must have the following structure:

```YAML
input_dir: preomop/03_omop_initial/
output_dir: preomop/04_omop_intermediate/VISIT/
input_files:
  - 02_Patologias_BPS.parquet
  - 03_MPA.parquet
  - 04_Hospitalizations.parquet
transformations: # Transformation to apply to each file
  - 02_Patologias_BPS.parquet: function_name
visit_concept_dict: # For each file
  02_Patologias_BPS.parquet:
    - - single_code # Apply this value for every row
      - 9202
      - {}
  03_MPA.parquet:
    - - single_code # Apply this value for every row
      - 32036
      - {}
  04_Hospitalizations.parquet:
    - - duration_code # Apply this code
      - 8756 # Outpatient hospital
      # When the number of days between end_date and start_date
      - time_lims: 
        - 0 # Has at least this value
        - 1 # Has at most this value
    - - duration_code # Apply this code
      - 8717 # Inpatient hospital
      # When the number of days between end_date and start_date
      - time_lims: 
        - 1 # Has at least this value
        - 730 # Has at most this value
visit_concept_order: # Hierarchy of concepts. When one of two rows have to be removed, the top one stays
  - 8717
  - 8756
  - 9202
  - 32036
  - 0

remove_overlap:
  sorting_columns: ["person_id", "start_date", "end_date", "visit_concept_id"]
  ascending_order: [True, True, False, True]

provider_table_path: preomop/04_omop_intermediate/PROVIDER/PROVIDER.parquet
source_to_provider_id:
  Hepatocarcinoma_Consultas_Externas.parquet: 
    COD_ESPECIALIDAD: specialty_source_value
```

The parameters are:
- `input_dir` is the path from `data_dir` to the directory where input data is.
- `output_dir` is the path from `data_dir` to the directory where data will be saved to.
- `input_files` is the list of files, as paths from `data_dir / input_dir`, to be used.
- `transformations`: Defines the transformations that must be made to the file before incorporating it to the rest of the data. 
  - A clear example would be one in which the column `end_date` is not reliable or relevant. So it is replaced by a copy of start_date.
  - Transformations are performed by functions that operate on pyarrow tables. See [table_transformer.py](../external/bps_to_omop/bps_to_omop/table_transformer.py).
- `visit_concept_dict`: Dictionary indicating the order of the codes to apply and how to apply them. Right now there are three options:
    - `single_code`: Applies one code to all cases.
        - When the file has only one code.
    - `field_code`: Applies a code only to cases where the `column` column has the value `column_value`.
        - Useful when the visit type is encoded in a column inside the file.
    - `duration_code`: Applies a code only to cases where the interval between end_date and start_date is within maximum and minimum value.
        - Useful when the type of visit depends on its duration. E.g. if it is a one day visit to the hospital, it is an outpatient, if it is more than one day, it is probably an inpatient.
    - It is important to note that the order matters. At first code 0 is assigned to all rows, then the functions are applied in order. Therefore, the last entry will take precedence in case it is applied to more than one record. 
- `visit_concept_order`: Hierarchy for the codes used. This is used when grouping visits. Multiday visits cannot overlap. In case two visits occur during the same period, one must contain the other. This order identifies which visit should be kept and which should be discarded.
- `provider_table_path`: is the path from `data_dir` to the directory where the PROVIDER table is.
- `source_to_provider_id`: Defines the correspondance between each input file and the provider_id key. I.e.: For each file, defines the column in the input table and the column in the PROVIDER table that it links to.
  - Example:
    - `Hepatocarcinoma_Consultas_Externas.parquet`: 
        `COD_ESPECIALIDAD`: `specialty_source_value`
    - This says that column `COD_ESPECIALIDAD` in `Hepatocarcinoma_Consultas_Externas.parquet` is the same as `specialty_source_value` column in the PROVIDER table.

The parameters `input_dir`, `output_dir` and `provider_table_path` are defined in relation to the `data_dir` folder defined in the `.env` file.